# DSAI cluster — the two control experiments

Run all cells top to bottom. ~3 GPU-hours, detached under `nohup`, so closing the tab is safe.

**What this settles.** Two claims in `paper/m2_m3_first_pass.md` rest on runs with no control arm:

1. *"The instruction's causal influence dies gradually inside the VLM."* Our control sweep — same
   sweep on a contrast with **no failure to recover** — shows the same decay, which would make the
   profile an artifact of causal proximity. This reruns it at **his trial count** so the comparison
   is apples-to-apples.
2. A **CUDA rerun of his M2 sweep**. His was MPS, where he measured attn/mlp sites disagreeing with
   CPU by up to 0.21. Same seed, so a backend difference shows up directly.

Neither is needed for the paper as now framed. Run this to make the repo's mechanistic claims
correct, not to unblock submission.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.used,utilization.gpu --format=csv

## 2. Clone / update repo

In [ ]:
import os, pathlib, subprocess

REPO = "https://github.com/mzkaell/vla-where-does-language-die.git"
BRANCH = "phase1-scaffold"

def repo_root(start):
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "src" / "models").exists():
            return d
    return None

root = repo_root(pathlib.Path.cwd())
if root is None:
    target = pathlib.Path.home() / "vla-where-does-language-die"
    if not target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(target)], check=True)
    root = target
os.chdir(root)
print("repo:", os.getcwd())
subprocess.run(["git", "pull", "--rebase", "-q"])
subprocess.run(["git", "log", "--oneline", "-1"])

## 3. Install (~5 min)

In [ ]:
# lerobot is PINNED: src/models/smolvla.py re-implements the forward pass against
# 0.6.0's internal signature and refuses to run on anything else.
!pip install -q -e ".[vla,dev]"
!pip install -q "lerobot[smolvla]==0.6.0" scikit-learn
!python -c "import importlib.metadata as m, torch; print('lerobot', m.version('lerobot'), '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 4. GitHub token

`getpass` keeps it out of the notebook file. Fine-grained PAT, *Contents: read and write* on this
repo only. The dry-run push verifies it immediately — discovering a bad token after three
GPU-hours is exactly the failure this avoids.

In [ ]:
import getpass, os, subprocess
tok = getpass.getpass("GitHub token (blank = do not push): ").strip()
if tok:
    subprocess.run(["git", "remote", "set-url", "origin",
        f"https://{tok}@github.com/mzkaell/vla-where-does-language-die.git"], check=True)
    os.environ.update(GIT_PUSH="1", GIT_NAME="mzkaell",
                      GIT_EMAIL="schmalzmichael50@gmail.com")
    r = subprocess.run(["git", "push", "--dry-run", "origin", "HEAD"],
                       capture_output=True, text=True)
    print("push check:", "OK" if r.returncode == 0 else "FAILED\n" + r.stderr)
else:
    os.environ["GIT_PUSH"] = "0"
    print("WARNING: nothing will be pushed. Download results/ before the session ends.")

## 5. Data (~20 min first time, cached after)

In [ ]:
!python scripts/download_data.py --all
!ls stimuli/*.jsonl 2>/dev/null || python scripts/build_pairs.py --suite libero_goal --n 400

## 6. Pick a usable GPU

Attempts a real allocation rather than trusting the memory column — under exclusive-process mode
a GPU showing free memory can still refuse.

In [ ]:
import os, subprocess

def usable(g):
    return subprocess.run(["python", "-c", "import torch; torch.zeros(1).cuda()"],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)),
        capture_output=True).returncode == 0

rows = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used",
                       "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.strip().splitlines()
cands = sorted((int(r.split(",")[1]), int(r.split(",")[0])) for r in rows)
GPU = next((g for _, g in cands if usable(g)), None)
print("no usable GPU — rerun this cell shortly" if GPU is None else f"using GPU {GPU}")

## 7. Launch

Resumable and push-as-you-go: a stage whose `metrics.json` already exists is skipped, and results
are committed the moment each stage finishes rather than at the end.

In [ ]:
import os, subprocess, textwrap

script = textwrap.dedent('''
set -uo pipefail
CK=k1000dai/smolvla_libero_finetune
say(){ echo ""; echo "=== [$(date -u +%H:%M:%S)] $* ==="; }
save(){ [ "${GIT_PUSH:-0}" = 1 ] || { echo "!! not pushing"; return 0; }
  git add -A results/ 2>/dev/null
  git diff --cached --quiet && { echo "(nothing new)"; return 0; }
  git -c user.name="$GIT_NAME" -c user.email="$GIT_EMAIL" commit -q -m "$1" \\
    && git push -q origin HEAD && echo "pushed: $1" || echo "!! PUSH FAILED"; }
have(){ [ -f "results/$1/metrics.json" ]; }

say "smoke test"
python scripts/run_localization.py --checkpoint $CK --device cuda \\
  --n-trials 2 --sites-limit 6 --null-sites 3 --min-trials 2 --resamples 500 --run-id _smoke
have _smoke || { echo "!! SMOKE FAILED - stopping before burning hours"; exit 1; }
rm -rf results/_smoke; echo "smoke OK"

if ! have locctl_anant_contrasts; then
  say "proximity control, n=40 to match loc_full_mps"
  python scripts/run_localization.py --checkpoint $CK --device cuda \\
    --n-trials 40 --contrast-mode control --run-id locctl_anant_contrasts
  save "M2 proximity control matching loc_full_mps trial count"
fi

if ! have loc_cuda_n40; then
  say "M2 sweep on CUDA (backend cross-check against his MPS run)"
  python scripts/run_localization.py --checkpoint $CK --device cuda \\
    --n-trials 40 --run-id loc_cuda_n40
  save "M2 sweep on CUDA for backend comparison"
fi

say "COMPLETE"; ls -1 results/; save "session results"
''')

with open("run_controls.sh", "w", newline="\n") as f:
    f.write(script)

subprocess.Popen("nohup bash run_controls.sh >> controls.log 2>&1 &", shell=True,
                 env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU)))
print("launched — safe to close the tab")

## 8. Monitor (re-run any time)

In [ ]:
!grep -E "^===|OK|FAIL|pushed" controls.log | tail -20

## 9. The answer

The number that matters is **novel minus control**. Near zero means the graded VLM decay in
`loc_full_mps` is causal proximity rather than binding, and conclusion 1 of
`m2_m3_first_pass.md` needs withdrawing.

In [ ]:
import json, pathlib

def prof(run, tower="vlm", comp="resid_post"):
    p = pathlib.Path(f"results/{run}/metrics.json")
    if not p.exists():
        return None
    S = {s["site"]: s for s in json.loads(p.read_text())["sites"]}
    return [S.get(f"{tower}.L{L}.{comp}", {}).get("recovery", {}).get("value")
            for L in range(16)]

nov, ctl = prof("loc_cuda_n40"), prof("locctl_anant_contrasts")
if nov and ctl:
    print(f"{'layer':>6}{'novel':>9}{'control':>9}{'diff':>9}")
    ds = []
    for L, (a, b) in enumerate(zip(nov, ctl)):
        if a is None or b is None:
            continue
        ds.append(a - b)
        print(f"{L:>6}{a:>9.3f}{b:>9.3f}{a - b:>+9.3f}")
    print(f"\nmean novel-minus-control: {sum(ds) / len(ds):+.3f}")
    print("near zero => the decay is proximity, not binding")
else:
    print("not finished yet — check the monitor cell")